# Paquetes de instalación

### Celda 1 — Instalación de librerías
Instala `ultralytics` (YOLO) y `supervision` en modo silencioso desde pip.

In [ ]:
!pip install ultralytics supervision -q
# !pip install roipoly

### Celda 2 — Importación de dependencias
Carga todas las librerías necesarias: torch, cv2, numpy, YOLO, dataclasses, entre otras.

In [ ]:
from google.colab import drive
import torch
import cv2
import numpy as np
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Optional
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import os


### Celda 3 — Montar Google Drive
Monta el Drive del usuario en `/content/drive` para acceder a archivos de video.

In [ ]:
drive.mount('/content/drive')

### Celda 4 — Verificación de GPU
Comprueba si CUDA está disponible e imprime el nombre del dispositivo (GPU o CPU).

In [ ]:
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### Celda 5 — Dataclasses del dominio
Define las estructuras de datos: `Detection`, `Track`, `SpatialTrack`, `StateChange`, `DirectionChange` y `Event`.

In [ ]:
@dataclass
class Detection:
    bbox: tuple
    confidence: float
    class_id: int = 0

@dataclass
class Track:
    id: int
    bbox: tuple

@dataclass
class SpatialTrack:
    id: int
    bbox: tuple
    inside: bool
    center: tuple[float, float]

@dataclass
class StateChange:
    track_id: int
    prev_inside: Optional[bool]
    curr_inside: bool
    direction: Optional[str] = None

@dataclass
class DirectionChange:
    track_id: int
    prev_point: tuple
    curr_point: tuple
    direction: str

@dataclass
class Event:
    track_id: int
    event_type: str

### Celda 6 — Interfaces abstractas (ABC)
Declara los contratos base: `Detector`, `Tracker`, `RoiEngine`, `StateManager`, `EventEngine` y `Visualizer`.

In [ ]:
class Detector(ABC):
    @abstractmethod
    def detect(self, frame: np.ndarray) -> list[Detection]: ...

class Tracker(ABC):
    @abstractmethod
    def track(self, detections: list[Detection], frame: np.ndarray) -> list[Track]: ...

class RoiEngine(ABC):
    @abstractmethod
    def is_inside(self, point: tuple[float, float]) -> bool: ...

class StateManager(ABC):
    @abstractmethod
    def update(self, tracks: list[SpatialTrack]) -> list[StateChange]: ...

class EventEngine(ABC):
    @abstractmethod
    def process(self, changes: list[StateChange]) -> list[Event]: ...

class Visualizer(ABC):
    @abstractmethod
    def draw(self, frame: np.ndarray, spatial_tracks: list[SpatialTrack],
             events: list[Event]) -> np.ndarray: ...

### Celda 7 — YoloDetector
Implementa el detector de personas usando un modelo YOLO; filtra por clase 0 (persona) con un umbral de confianza.

In [ ]:
class YoloDetector(Detector):
    def __init__(self, model_path: str = "yolo11n.pt", conf: float = 0.3):
        self.model = YOLO(model_path)
        self.conf = conf

    def detect(self, frame: np.ndarray) -> list[Detection]:
        results = self.model(frame, conf=self.conf, classes=[0], verbose=False)[0]
        detections = []
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            detections.append(Detection(
                bbox=(x1, y1, x2, y2),
                confidence=float(box.conf[0]),
                class_id=int(box.cls[0])
            ))
        return detections

### Celda 8 — ByteTrackTracker
Implementa el tracker usando YOLO + ByteTrack; asigna un ID persistente a cada persona detectada.

In [ ]:
class ByteTrackTracker(Tracker):
    def __init__(self, model_path: str = "yolo11n.pt", conf: float = 0.3):
        self.model = YOLO(model_path)
        self.conf = conf

    def track(
        self,
        detections: list[Detection],
        frame: np.ndarray
    ) -> list[Track]:

        results = self.model.track(
            frame,
            conf=self.conf,
            classes=[0],
            tracker="bytetrack.yaml",
            persist=True,
            verbose=False
        )[0]

        tracks = []

        if results.boxes.id is not None:

            for box, track_id in zip(
                results.boxes,
                results.boxes.id
            ):

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                conf = float(box.conf[0])

                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2

                tracks.append(
                    Track(
                        id=int(track_id),
                        bbox=(x1, y1, x2, y2),
                    )
                )

        return tracks

### Celda 9 — OpenCvRoiEngine
Define una ROI poligonal y determina si un punto está dentro usando `cv2.pointPolygonTest`.

In [ ]:
class OpenCvRoiEngine(RoiEngine):
    def __init__(self, polygon: list[tuple[int, int]]):
        # El polígono se define como lista de puntos (x, y) en píxeles del frame
        self.polygon = np.array(polygon, dtype=np.int32)

    def is_inside(self, point: tuple[float, float]) -> bool:
        # >= 0 significa dentro o sobre el borde del polígono
        result = cv2.pointPolygonTest(self.polygon, point, measureDist=False)
        return result >= 0

### Celda 10 — InMemoryStateManager
Mantiene en memoria el estado (dentro/fuera) y la posición/dirección de cada track, detectando cambios de estado.

In [ ]:
class InMemoryStateManager(StateManager):
    def __init__(self, history_len: int = 10):
        self._state: dict[int, bool] = {}
        self._positions = {}
        self._directions = {}
        self._direction_history: dict[int, list[str]] = {}
        self._history_len = history_len

    def update(self, tracks: list[SpatialTrack]) -> list[StateChange]:

        changes = []
        current_ids = {t.id for t in tracks}

        for track in tracks:

            prev_pos = self._positions.get(track.id)

            if prev_pos is not None:

                direction = get_direction(
                    prev_pos,
                    track.center
                )

                self._directions[track.id] = direction

                if direction != "quieto":
                    hist = self._direction_history.setdefault(track.id, [])
                    hist.append(direction)
                    if len(hist) > self._history_len:
                        hist.pop(0)

            self._positions[track.id] = track.center

            prev = self._state.get(track.id)

            if prev != track.inside:

                hist = self._direction_history.get(track.id, [])
                dominant = max(set(hist), key=hist.count) if hist else self._directions.get(track.id)

                changes.append(
                    StateChange(
                        track_id=track.id,
                        prev_inside=prev,
                        curr_inside=track.inside,
                        direction=dominant
                    )
                )

            self._state[track.id] = track.inside

        # Limpiar tracks desaparecidos
        for tid in set(self._state) - current_ids:
            del self._state[tid]

        for tid in set(self._positions) - current_ids:
            del self._positions[tid]

        for tid in set(self._directions) - current_ids:
            del self._directions[tid]

        for tid in set(self._direction_history) - current_ids:
            del self._direction_history[tid]

        return changes

### Celda 11 — DefaultEventEngine
Convierte los cambios de estado en eventos (`entered`, `exited`, `appeared_inside`) e incrementa los contadores globales.

In [ ]:
class DefaultEventEngine(EventEngine):
    def process(self, changes: list[StateChange]) -> list[Event]:

        global contador_suben
        global contador_bajan
        global contador_roi

        events = []

        for c in changes:

            if c.prev_inside is None and c.curr_inside:
                event_type = "appeared_inside"

            elif c.prev_inside is False and c.curr_inside:

                contador_roi += 1

                if c.direction == "subiendo":
                    contador_suben += 1

                elif c.direction == "bajando":
                    contador_bajan += 1

                event_type = "entered"

            elif c.prev_inside is True and not c.curr_inside:
                event_type = "exited"

            else:
                continue

            events.append(
                Event(
                    track_id=c.track_id,
                    event_type=event_type
                )
            )

        return events

### Celda 12 — OpenCvVisualizer
Dibuja sobre cada frame: la ROI, los bounding boxes con IDs/eventos y los contadores de suben/bajan/total.

In [ ]:
class OpenCvVisualizer(Visualizer):
    def __init__(self, roi_polygon: list[tuple[int, int]]):
        self.roi_polygon = np.array(roi_polygon, dtype=np.int32)

    def draw(self, frame: np.ndarray, spatial_tracks: list[SpatialTrack],
             events: list[Event]) -> np.ndarray:
        out = frame.copy()
        cv2.polylines(out, [self.roi_polygon], isClosed=True, color=(0, 0, 255), thickness=2)

        event_map = {e.track_id: e.event_type for e in events}

        for st in spatial_tracks:
          x1, y1, x2, y2 = map(int, st.bbox)
          color = (0, 255, 0) if st.inside else (0, 0, 255)

          cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)

          label = f"ID:{st.id}"

          if st.id in event_map:
              label += f" [{event_map[st.id]}]"

          cv2.putText(
              out,
              label,
              (x1, y1 - 8),
              cv2.FONT_HERSHEY_SIMPLEX,
              0.5,
              color,
              1
          )


        stats = [
            (f"SUBEN: {contador_suben}", (20, 40)),
            (f"BAJAN: {contador_bajan}", (20, 80)),
            (f"TOTAL ROI: {contador_roi}", (20, 120)),
        ]
        for text, (x, y) in stats:
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
            cv2.rectangle(out, (x - 4, y - th - 2), (x + tw + 4, y + 6), (255, 255, 255), -1)
            cv2.putText(out, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
        return out


### Celda 13 — TrackingPipeline
Orquesta el flujo completo: tracking → ROI → estado → eventos → visualización, frame a frame.

In [ ]:
class TrackingPipeline:
    def __init__(self, tracker, roi, state, events, visualizer):
        self.tracker = tracker
        self.roi = roi
        self.state = state
        self.events = events
        self.visualizer = visualizer

    def process(self, frame: np.ndarray):
        tracks = self.tracker.track([], frame)


        spatial_tracks = [
            SpatialTrack(
                id=t.id,
                bbox=t.bbox,
                inside=self.roi.is_inside(self._foot_point(t.bbox)),
                center=self._center_point(t.bbox)
            )
            for t in tracks
        ]

        changes = self.state.update(spatial_tracks)
        evts = self.events.process(changes)
        annotated = self.visualizer.draw(frame, spatial_tracks, evts)
        return annotated, evts

    @staticmethod
    def _foot_point(bbox) -> tuple[float, float]:
        # Centro del borde inferior del bbox
        x1, _, x2, y2 = bbox
        return ((x1 + x2) / 2, y2)
    @staticmethod
    def _center_point(bbox) -> tuple[float, float]:
        x1, y1, x2, y2 = bbox
        return ((x1 + x2) / 2, (y1 + y2) / 2)


### Celda 14 — Configuración e inicialización del pipeline
Define paths, el polígono ROI, la función de dirección y los contadores; luego instancia el pipeline completo.

In [ ]:
VIDEO_PATH = "/content/sample_data/sample-p2.mp4"
OUTPUT_PATH = "output.mp4"

# Polígono de la ROI — Modificamos estos puntos según la escena
ROI_POLYGON = [
    [1824, 500],
    [1904, 436],
    [1122, 332],
    [984, 366]
]

STAIR_START = np.array([1864, 468])
STAIR_END   = np.array([1053, 349])

def get_direction(prev_point, curr_point):

    stair_vector = STAIR_END - STAIR_START  # apunta hacia izquierda/arriba

    movement = np.array(curr_point) - np.array(prev_point)

    projection = np.dot(movement, stair_vector)

    if projection > 0:
        return "bajando"    # moverse hacia STAIR_END = izquierda = bajar

    elif projection < 0:
        return "subiendo"   # moverse contra STAIR_END = derecha = subir

    return "quieto"

#Contadores globales
contador_suben = 0
contador_bajan = 0
contador_roi = 0

pipeline = TrackingPipeline(
    tracker=ByteTrackTracker("yolo11n.pt", conf=0.3),
    roi=OpenCvRoiEngine(ROI_POLYGON),
    state=InMemoryStateManager(),
    events=DefaultEventEngine(),
    visualizer=OpenCvVisualizer(ROI_POLYGON),
)
print("Pipeline listo ✓")

### Celda 15 — Reescalado de video a 1080p con FFmpeg
Convierte el video de entrada a 1920×1080 con padding, usando codec H.264, y guarda el resultado.

In [ ]:
INPUT_PATH = "/content/sample_data/sample-p2.mp4"
VIDEO_PATH = "/content/sample_data/sample-p2_1080p.mp4"  # archivo nuevo, no pisa el original

!ffmpeg -i {INPUT_PATH} \
    -vf "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2" \
    -vcodec libx264 -crf 23 -preset fast \
    -an \
    {VIDEO_PATH} -y

import os
size = os.path.getsize(VIDEO_PATH)
print(f"Output: {size/1024/1024:.1f} MB")

### Celda 16 — Loop principal de procesamiento
Recorre el video frame a frame, aplica el pipeline y escribe el video anotado; al final lo recodifica a H.264.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

# Verificar que el video se abrió correctamente antes de entrar al loop
assert cap.isOpened(), f"No se pudo abrir el video: {VIDEO_PATH}"

h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {w}x{h} @ {fps}fps — {total_frames} frames")

# Inicializar el writer ANTES del loop con las dimensiones conocidas
out_writer = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (w, h)
)

assert out_writer.isOpened(), "El VideoWriter no se pudo abrir — revisá el codec/fourcc"

total_events = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    annotated, events = pipeline.process(frame)
    out_writer.write(annotated)

    if events:
        for e in events:
            print(f"[EVENT] track={e.track_id} → {e.event_type}")
            total_events.append(e)

cap.release()
out_writer.release()
print(f"\nDone ✓ — {len(total_events)} eventos totales → {OUTPUT_PATH}")

OUTPUT_H264_PATH = "output_h264.mp4"
!ffmpeg -i {OUTPUT_PATH} -vcodec libx264 -crf 23 -preset fast -pix_fmt yuv420p {OUTPUT_H264_PATH} -y

size = os.path.getsize(OUTPUT_H264_PATH) if os.path.exists(OUTPUT_H264_PATH) else 0
print(f"\nVideo final reproducible → {OUTPUT_H264_PATH} ({size/1024/1024:.1f} MB)")

### Celda 17 — Selector de ROI interactivo (comentado)
Código desactivado que permitía dibujar la ROI sobre un frame del video usando un canvas HTML con JavaScript.

In [ ]:
# # ============================================================
# # CELDA — ROI selector con ipywidgets
# # ============================================================
# import cv2
# import numpy as np
# from IPython.display import display, clear_output
# import ipywidgets as widgets
# from PIL import Image
# import io
# import base64

# cap = cv2.VideoCapture(VIDEO_PATH)
# ret, frame = cap.read()
# cap.release()

# scale = 0.5
# preview = cv2.resize(frame, (0, 0), fx=scale, fy=scale)
# preview_rgb = cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)

# clicked_points = []

# def frame_to_base64(img_array):
#     pil_img = Image.fromarray(img_array)
#     buf = io.BytesIO()
#     pil_img.save(buf, format='PNG')
#     return base64.b64encode(buf.getvalue()).decode()

# def redraw():
#     img = preview_rgb.copy()
#     img_pil = Image.fromarray(img)
#     import PIL.ImageDraw
#     draw = PIL.ImageDraw.Draw(img_pil)
#     if len(clicked_points) >= 2:
#         pts = clicked_points + [clicked_points[0]]
#         draw.line(pts, fill=(255, 255, 0), width=3)
#     for p in clicked_points:
#         draw.ellipse([p[0]-5, p[1]-5, p[0]+5, p[1]+5], fill=(255, 255, 0))
#     img_widget.value = img_pil._repr_png_()  # no funciona así

# # Alternativa más simple: usar HTML + JavaScript
# from IPython.display import HTML

# h, w = preview_rgb.shape[:2]
# b64 = frame_to_base64(preview_rgb)

# html_code = f"""
# <canvas id="roiCanvas" width="{w}" height="{h}" style="cursor:crosshair; border:1px solid #ccc;"></canvas>
# <br>
# <button onclick="clearPoints()">Limpiar</button>
# <button onclick="printPoints()">Confirmar ROI</button>
# <pre id="output"></pre>

# <script>
# var canvas = document.getElementById('roiCanvas');
# var ctx = canvas.getContext('2d');
# var points = [];
# var img = new Image();
# img.src = 'data:image/png;base64,{b64}';
# img.onload = function() {{ ctx.drawImage(img, 0, 0); }};

# canvas.addEventListener('click', function(e) {{
#     var rect = canvas.getBoundingClientRect();
#     var x = Math.round(e.clientX - rect.left);
#     var y = Math.round(e.clientY - rect.top);
#     points.push([x, y]);
#     ctx.clearRect(0, 0, canvas.width, canvas.height);
#     ctx.drawImage(img, 0, 0);
#     ctx.strokeStyle = 'yellow';
#     ctx.lineWidth = 2;
#     ctx.fillStyle = 'yellow';
#     if (points.length > 1) {{
#         ctx.beginPath();
#         ctx.moveTo(points[0][0], points[0][1]);
#         for (var i = 1; i < points.length; i++) ctx.lineTo(points[i][0], points[i][1]);
#         ctx.closePath();
#         ctx.stroke();
#     }}
#     points.forEach(function(p) {{
#         ctx.beginPath();
#         ctx.arc(p[0], p[1], 5, 0, 2*Math.PI);
#         ctx.fill();
#     }});
#     document.getElementById('output').innerText = 'Puntos (preview): ' + JSON.stringify(points);
# }});

# function clearPoints() {{
#     points = [];
#     ctx.clearRect(0, 0, canvas.width, canvas.height);
#     ctx.drawImage(img, 0, 0);
#     document.getElementById('output').innerText = '';
# }}

# function printPoints() {{
#     var scale = {scale};
#     var real = points.map(p => [Math.round(p[0]/scale), Math.round(p[1]/scale)]);
#     document.getElementById('output').innerText = 'ROI_POLYGON = ' + JSON.stringify(real);
# }}
# </script>
# """

# display(HTML(html_code))

### Celda 18 — Previsualización de la ROI sobre el video
Toma el primer frame del video, dibuja el polígono ROI en cyan y lo muestra reducido a 960×540.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()

# Dibujar la ROI actual para ver dónde queda
roi_pts = np.array(ROI_POLYGON, dtype=np.int32)
preview = frame.copy()
cv2.polylines(preview, [roi_pts], isClosed=True, color=(0, 0, 255), thickness=4)

# Reducir para que entre en el notebook
preview_small = cv2.resize(preview, (960, 540))
cv2_imshow(preview_small)

### Celda 19 — Listado de archivos MP4 generados
Lista todos los archivos `.mp4` en `/content` mostrando su nombre y tamaño en MB.

In [ ]:
for f in os.listdir("/content"):
    if f.endswith(".mp4"):
        size = os.path.getsize(f"/content/{f}")
        print(f"{f} → {size/1024/1024:.1f} MB")